# 05 — Size scaling

**Sweep 2.** Vary `n in {4, 6, 8, 10, 12}` at fixed `p=3`, fixed `K/n ratio`.
Honest answer at these sizes: classical methods are strictly better — faster,
more reliable, no barren plateaus. QAOA matches optimality but doesn't
outperform. The methodology is the point.

In [ ]:
# === Bootstrap (Colab + local) ===
import sys, os, json
try:
    import google.colab  # noqa: F401
    get_ipython().system('test -d /content/fys5419 || git clone -q https://github.com/egil10/fys5419.git /content/fys5419')
    get_ipython().run_line_magic('cd', '/content/fys5419/project2/code/notebooks')
except ImportError:
    pass
sys.path.append('..')
from scripts.colab import setup; setup()

# === Project imports ===
from pathlib import Path
import numpy as np
import pandas as pd

from scripts.data      import load_returns
from scripts.portfolio import PortfolioProblem
from scripts.classical import brute_force, greedy_top_k, simulated_annealing, markowitz_round
from scripts.qaoa      import solve
from scripts.metrics   import prob_optimal

RESULTS = Path.cwd().parent / 'results'
RESULTS.mkdir(exist_ok=True)

In [ ]:
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'IBM', 'HON', 'ACN',
            'XOM',  'JPM',  'DAL',   'NEM']
START, END = '2023-01-01', '2025-12-31'

N_VALUES = [4, 6, 8, 10, 12]
P        = 3
K_FRAC   = 0.25
LAM, AP  = 2.0, 0.5
N_SEEDS  = 10

In [ ]:
cache = RESULTS / 'size_scaling.json'

if cache.exists():
    rows = json.loads(cache.read_text())
    print(f'loaded {cache.name}')
else:
    rows = []
    for n in N_VALUES:
        tickers = UNIVERSE[:n]
        K = max(1, round(K_FRAC * n))
        r  = load_returns(tickers, START, END, cache_name=f'scaling_n{n}')
        pf = PortfolioProblem(r.mu, r.Sigma, lam=LAM, A=AP, K=K, tickers=tuple(r.tickers))

        bf = brute_force(pf)

        # Multi-seed SA: report median
        sa_runs = [simulated_annealing(pf, n_sweeps=1000*n, seed=s) for s in range(N_SEEDS)]
        sa_med  = float(np.median([sr.cost for sr in sa_runs]))
        sa_t    = float(np.median([sr.runtime for sr in sa_runs]))

        # QAOA at fixed p
        qres = solve(pf, p=P, n_restarts=N_SEEDS, seed=42)

        rows.extend([
            {'n': n, 'K': K, 'solver': 'brute_force',     'cost': bf.cost,                            'runtime_s': bf.runtime,                        'ratio': 1.0},
            {'n': n, 'K': K, 'solver': 'greedy_sharpe',   'cost': greedy_top_k(pf).cost,              'runtime_s': greedy_top_k(pf).runtime,         'ratio': greedy_top_k(pf).cost / bf.cost},
            {'n': n, 'K': K, 'solver': 'markowitz_round', 'cost': markowitz_round(pf).cost,           'runtime_s': markowitz_round(pf).runtime,      'ratio': markowitz_round(pf).cost / bf.cost},
            {'n': n, 'K': K, 'solver': 'sa_median',       'cost': sa_med,                             'runtime_s': sa_t,                              'ratio': sa_med / bf.cost},
            {'n': n, 'K': K, 'solver': f'qaoa_p{P}',      'cost': float(qres['energy']),              'runtime_s': qres['runtime'],                   'ratio': float(qres['energy']) / bf.cost, 'p_optimal': prob_optimal(qres['probs'], bf.x)},
        ])
        print(f'  n={n} K={K}: brute={bf.cost:.4f}  sa_med={sa_med:.4f}  qaoa={qres["energy"]:.4f}')

    cache.write_text(json.dumps(rows, indent=2))
    print(f'saved -> {cache.name}')

pd.DataFrame(rows)